In [ ]:
!pip install chromadb
!pip install dspy-ai
!pip install groq
!pip install -U sentence-transformers
!pip install langchain
!pip install -U langchain-community
!pip install gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.5/559.5 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 10.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 77.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.0/107.0 kB 15.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 10.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.7/283.7 kB 28.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 

In [ ]:
import chromadb
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
import dspy
from dspy.retrieve.chromadb_rm import ChromadbRM
from sklearn.model_selection import train_test_split
import langchain
from langchain.document_loaders import CSVLoader
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from langchain.text_splitter import RecursiveCharacterTextSplitter, SentenceTransformersTokenTextSplitter
import regex as re
from groq import Groq

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [ ]:
os.environ['GROQ_API_KEY'] = ''

### Converting csv to document

In [ ]:
df=pd.read_csv('final.csv')
# df.drop(columns=['Unnamed: 0'],inplace=True)

In [ ]:
df.head()

,question,answer
0,Is there a review of Grand Park Hotel in New Y...,The Grand Park Hotel exceeded all my expectati...
1,What is the location of Grand Park Hotel?,"New York City, USA"
2,What is the most memorable experience you had ...,The Grand Park Hotel exceeded all my expectati...
3,What is the review of Sunset Beach Resort in P...,I had an amazing experience at the Sunset Beac...
4,What is the location of Sunset Beach Resort?,"Phuket, Thailand"


In [ ]:
loader = CSVLoader("./final.csv", encoding="windows-1252")
documents = loader.load()

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
documents[0].page_content

": 0\nquestion: Is there a review of Grand Park Hotel in New York City?\nanswer: The Grand Park Hotel exceeded all my expectations. The staff was friendly and accommodating, and the rooms were luxurious and comfortable. The hotel's location in the heart of New York City made it convenient to explore the city's attractions. I would highly recommend this hotel to anyone visiting New York City."

### Creating Dspy custom dataset

In [ ]:
dataset = []

for question, answer in df.values:
    dataset.append(dspy.Example(question=question, answer=answer).with_inputs("question"))

print(dataset[:3])

[Example({'question': 'Is there a review of Grand Park Hotel in New York City?', 'answer': "The Grand Park Hotel exceeded all my expectations. The staff was friendly and accommodating, and the rooms were luxurious and comfortable. The hotel's location in the heart of New York City made it convenient to explore the city's attractions. I would highly recommend this hotel to anyone visiting New York City."}) (input_keys={'question'}), Example({'question': 'What is the location of Grand Park Hotel?', 'answer': 'New York City, USA'}) (input_keys={'question'}), Example({'question': 'What is the most memorable experience you had at the Grand Park Hotel?', 'answer': "The Grand Park Hotel exceeded all my expectations. The staff was friendly and accommodating, and the rooms were luxurious and comfortable. The hotel's location in the heart of New York City made it convenient to explore the city's attractions. I would highly recommend this hotel to anyone visiting New York City."}) (input_keys={'q

### Splitting the data and creating chunks,embeddings and ingestion to chromadb

In [ ]:
# Perform the train-test split
train_df, test_df = train_test_split(dataset, test_size=0.2, random_state=42)

# Display the shapes of the resulting DataFrames
print(f"Train DataFrame shape: {len(train_df)}")
print(f"Test DataFrame shape: {(test_df)}")

Train DataFrame shape: 252
Test DataFrame shape: 63


In [ ]:
train_df[:2]


[Example({'question': "What are the hotel's amenities?", 'answer': "Golden Sands Hotel provided a fantastic experience in Miami. The beachfront location was perfect, and the views were stunning. The rooms were spacious, clean, and well-appointed. The staff was friendly, attentive, and provided exceptional service. The hotel's amenities, including the pool and beach access, were excellent. I would highly recommend Golden Sands Hotel for a memorable stay in Miami."}) (input_keys={'question'}),
 Example({'question': 'What is the review of Mountain View Resort in Lake Tahoe, USA?', 'answer': "Mountain View Resort provided a spectacular mountain retreat in Lake Tahoe. The resort's location amidst the towering Sierra Nevada mountains and crystal-clear lake was breathtaking. The rooms were spacious, elegantly furnished, and offered stunning views of the surrounding nature. The staff was warm, attentive, and provided excellent recommendations for outdoor activities such as hiking and skiing. A

In [ ]:
texts=[text.page_content for text in documents]
texts[0]

": 0\nquestion: Is there a review of Grand Park Hotel in New York City?\nanswer: The Grand Park Hotel exceeded all my expectations. The staff was friendly and accommodating, and the rooms were luxurious and comfortable. The hotel's location in the heart of New York City made it convenient to explore the city's attractions. I would highly recommend this hotel to anyone visiting New York City."

In [ ]:
#step 1
character_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=1000,
    chunk_overlap=0
)
character_split_texts = character_splitter.split_text('\n\n'.join(texts))


In [ ]:
print((character_split_texts[10]))
print(f"\nTotal chunks: {len(character_split_texts)}")

: 20
question: What was the most memorable experience you had at Seaside Paradise Resort?
answer: My vacation at Seaside Paradise Resort was nothing short of amazing. The resort's prime beachfront location offered crystal-clear turquoise waters and pristine white sand. The rooms were spacious, clean, and tastefully decorated. The resort's pool area was a tropical oasis, complete with palm trees and refreshing cocktails. The staff was attentive and friendly, ensuring that every aspect of my stay was perfect. I would return to Seaside Paradise Resort in a heartbeat.

Total chunks: 184


In [ ]:
#step 2
#to ensure data is within embeddings context window length
token_splitter = SentenceTransformersTokenTextSplitter(chunk_overlap=0, tokens_per_chunk=256)

token_split_texts = []
for text in character_split_texts:
    token_split_texts += token_splitter.split_text(text)

print((token_split_texts[10]))
print(f"\nTotal chunks: {len(token_split_texts)}")

: 20 question : what was the most memorable experience you had at seaside paradise resort? answer : my vacation at seaside paradise resort was nothing short of amazing. the resort's prime beachfront location offered crystal - clear turquoise waters and pristine white sand. the rooms were spacious, clean, and tastefully decorated. the resort's pool area was a tropical oasis, complete with palm trees and refreshing cocktails. the staff was attentive and friendly, ensuring that every aspect of my stay was perfect. i would return to seaside paradise resort in a heartbeat.

Total chunks: 184


In [ ]:
embedding_function = SentenceTransformerEmbeddingFunction()

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/chroma")
chroma_collection = chroma_client.create_collection("hoteldata", embedding_function=embedding_function)

ids = [str(i) for i in range(len(token_split_texts))]

chroma_collection.add(ids=ids, documents=token_split_texts)
chroma_collection.count()

184

In [ ]:
retriever_model = ChromadbRM(
    'hoteldata',
    '/content/drive/MyDrive/chroma',
    embedding_function=embedding_function,
    k=3
)

### configuring dspy

In [ ]:
llama3 = dspy.GROQ(model='llama3-8b-8192', api_key=os.environ['GROQ_API_KEY'])

In [ ]:
dspy.settings.configure(rm=retriever_model, lm=llama3)

In [ ]:
class GenerateAnswer(dspy.Signature):
    """Answer questions with short relevant answers."""

    context = dspy.InputField(desc="may contain relevant information")
    question = dspy.InputField()
    answer = dspy.OutputField(desc="give a recommendation between 10-15 words")
    # reason = dspy.OutputField(desc="give relevant,logical reasoning for the recommendation")

In [ ]:
class RAG(dspy.Module):
    def __init__(self, num_passages=3):
        super().__init__()

        self.retrieve = dspy.Retrieve(k=num_passages)
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question):
        context = self.retrieve(question).passages
        prediction = self.generate_answer(context=context, question=question)
        return dspy.Prediction(context=context, answer=prediction.answer)

In [ ]:
class Assess(dspy.Signature):
    """Assess the quality of an answer along the specified dimension."""
    assessed_text = dspy.InputField()
    assessment_question = dspy.InputField()
    assessment_answer = dspy.OutputField(desc="Only output an integer score between 0-10 and nothing else")

def extract_score(text):
    matches = re.findall(r'\d+', text)
    if matches:
        return int(matches[-1])
    else:
        return -1  # Default value if no integer is found

def metric(example, pred, trace=None):
    question, answer, context, prediction = example.question, example.answer, pred.context, pred.answer
    #converting list to str
    context = " ".join(context)
    # print(context[:200])

    # Content relevance assessment question
    content_question =f"Does the assessed text answer the question '{question}' deriving from or similar to '{answer}'?"
    # # Correctness assessment question while passing context length of 200 characters to ensure assessment answer for the query
    correct_question = f"Is the assessed text relevant to {context[:200]}?"
    # exact_question = f"Is the assessed similar to'{answer}'?"

    with dspy.context(lm=llama3):
        # Ensure questions are passed as strings
        content_assessment = dspy.Predict(Assess)(assessed_text=prediction, assessment_question=content_question)
        correct_assessment = dspy.Predict(Assess)(assessed_text=prediction, assessment_question=correct_question)



    content_score=extract_score(content_assessment.assessment_answer)
    correct_score=extract_score(correct_assessment.assessment_answer)
    # print(content_score)
    # print(correct_score)
    score=(content_score+correct_score)
    if trace is not None:
        return score >= 0.5
    return score / 20.0



In [ ]:
#optimize
from dspy.teleprompt import BootstrapFewShotWithRandomSearch

config = dict(max_bootstrapped_demos=4, max_labeled_demos=4, num_candidate_programs=2, num_threads=4)
# Set up teleprompter which will compile our RAG program
teleprompter = BootstrapFewShotWithRandomSearch(metric=metric,**config)

# Compile!
compiled_rag = teleprompter.compile(RAG(), trainset=train_df)

Average Metric: 3.9500000000000006 / 9  (43.9):   4%|▎         | 9/252 [00:06<03:51,  1.05it/s]INFO:backoff:Backing off request(...) for 0.9s (groq.RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama3-8b-8192` in organization `org_01j1skxws6et3tfhe3vj2hffj6` on requests per minute (RPM): Limit 30, Used 30, Requested 1. Please try again in 1.889s. Visit https://console.groq.com/docs/rate-limits for more information.', 'type': 'requests', 'code': 'rate_limit_exceeded'}})
INFO:backoff:Backing off request(...) for 1.0s (groq.RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama3-8b-8192` in organization `org_01j1skxws6et3tfhe3vj2hffj6` on requests per minute (RPM): Limit 30, Used 30, Requested 1. Please try again in 1.845s. Visit https://console.groq.com/docs/rate-limits for more information.', 'type': 'requests', 'code': 'rate_limit_exceeded'}})


Backing off 0.9 seconds after 1 tries calling function <function GroqLM.request at 0x7a83ec50ba30> with kwargs {}
Backing off 1.0 seconds after 1 tries calling function <function GroqLM.request at 0x7a83ec50ba30> with kwargs {}


INFO:backoff:Backing off request(...) for 0.3s (groq.RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama3-8b-8192` in organization `org_01j1skxws6et3tfhe3vj2hffj6` on requests per minute (RPM): Limit 30, Used 30, Requested 1. Please try again in 1.886s. Visit https://console.groq.com/docs/rate-limits for more information.', 'type': 'requests', 'code': 'rate_limit_exceeded'}})


Backing off 0.3 seconds after 1 tries calling function <function GroqLM.request at 0x7a83ec50ba30> with kwargs {}


INFO:backoff:Backing off request(...) for 0.4s (groq.RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama3-8b-8192` in organization `org_01j1skxws6et3tfhe3vj2hffj6` on requests per minute (RPM): Limit 30, Used 30, Requested 1. Please try again in 1.976s. Visit https://console.groq.com/docs/rate-limits for more information.', 'type': 'requests', 'code': 'rate_limit_exceeded'}})


...

Average Metric: 102.99999999999991 / 246  (41.9):  98%|█████████▊| 246/252 [28:14<00:43,  7.17s/it]INFO:backoff:Backing off request(...) for 2.7s (groq.RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama3-8b-8192` in organization `org_01j1skxws6et3tfhe3vj2hffj6` on requests per minute (RPM): Limit 30, Used 30, Requested 1. Please try again in 1.882s. Visit https://console.groq.com/docs/rate-limits for more information.', 'type': 'requests', 'code': 'rate_limit_exceeded'}})


Backing off 2.7 seconds after 3 tries calling function <function GroqLM.request at 0x7a83ec50ba30> with kwargs {}


Average Metric: 103.54999999999991 / 248  (41.8):  98%|█████████▊| 248/252 [28:28<00:28,  7.05s/it]INFO:backoff:Backing off request(...) for 0.4s (groq.RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama3-8b-8192` in organization `org_01j1skxws6et3tfhe3vj2hffj6` on requests per minute (RPM): Limit 30, Used 30, Requested 1. Please try again in 1.912s. Visit https://console.groq.com/docs/rate-limits for more information.', 'type': 'requests', 'code': 'rate_limit_exceeded'}})


Backing off 0.4 seconds after 1 tries calling function <function GroqLM.request at 0x7a83ec50ba30> with kwargs {}


Average Metric: 105.6999999999999 / 252  (41.9): 100%|██████████| 252/252 [29:49<00:00,  7.10s/it]


In [ ]:
compiled_rag.save('/content/drive/MyDrive/chroma/final_optimizer.json')

In [ ]:
compiled_rag=RAG()
compiled_rag.load(path='/content/drive/MyDrive/chroma/final_optimizer.json')

In [ ]:
#evaluate
from dspy.evaluate.evaluate import Evaluate
# Set up the `evaluate_on_qa` function. We'll use this many times below.
evaluate_on_qa = Evaluate(devset=test_df, num_threads=1, display_progress=True, display_table=5)


evaluate_on_qa(compiled_rag, metric=metric)

Average Metric: 27.24999999999999 / 63  (43.3): 100%|██████████| 63/63 [07:03<00:00,  6.71s/it]


,question,example_answer,context,pred_answer,metric
0,What could Harmony Retreat do to improve its services and guest experience?,I had a disappointing experience at Harmony Retreat. The rooms were small and poorly maintained. The staff was unprofessional and inattentive to guests' needs. The...,"["": 203 question : what is the most popular spa treatment offered at harmony retreat? answer : harmony retreat exceeded my expectations in every way....","Improve room maintenance, staff professionalism, and spa treatments.",✔️ [0.5]
1,Is there a Golden Valley Inn in Napa Valley?,My visit to the Golden Valley Inn in Napa Valley was absolutely delightful. The inn's picturesque location amidst vineyards and rolling hills was a sight...,"["": 180 question : is there a golden valley inn in napa valley? answer : my visit to the golden valley inn in napa valley...","Yes, there is a Golden Valley Inn in Napa Valley.",✔️ [1.0]
2,Which city is the Royal Mirage Palace located in?,"Marrakech, Morocco","["": 162 question : is the review of royal mirage palace in marrakech, morocco, positive? answer : staying at the royal mirage palace was a...","Marrakech, Morocco",✔️ [0.5]
3,"What is the review of Casa Bella Inn in Barcelona, Spain?",Casa Bella Inn provided a charming and authentic experience in Barcelona. The inn's traditional Spanish architecture and warm ambiance made me feel right at home....,"["": 78 question : what is the review of casa bella inn in barcelona, spain? answer : casa bella inn provided a charming and authentic...","Positive review, with the reviewer praising the inn's traditional Spanish architecture, warm ambiance, cozy rooms, friendly staff, and central location.",✔️ [0.45]
4,"What was the overall experience of staying at Island Paradise Resort in Phuket, Thailand?",My stay at Island Paradise Resort in Phuket was absolutely incredible. The resort's tropical setting and stunning beachfront location offered a true slice of paradise....,"["": 251 question : what was the most memorable experience you had at island paradise resort? answer : my stay at island paradise resort was...","Mixed reviews, with some praising the resort's location, amenities, and service, while others were disappointed with the cleanliness and service.",✔️ [0.05]


43.25

### Chatbot

In [ ]:
client = Groq(
    api_key= os.environ['GROQ_API_KEY'],
)
def llama3(query):
  chat_completion = client.chat.completions.create(
      messages=[
          {
              "role": "user",
              "content": query,
          }
      ],
      model="llama3-70b-8192",
  )
  return chat_completion.choices[0].message.content
# print(response('hi'))

In [ ]:
import gradio as gr
import time

def chatbot_interface(user_input, chat_history=[]):
    response = compiled_rag(user_input)

    # if (60 < len(response.answer) < 250) and ('reason' not in response.answer):
    if (60 < len(response.answer) < 250) and ('reason' or 'question seems to be missing') not in response.answer.lower():
        bot_response = "RAG: " + response.answer
    else:
        bot_response = "L3: " + llama3(f"Answer concisely and politely by considering all the information from {response} to respond to the question: {user_input}. Make use of {chat_history[:300]} if it exists.")

    chat_history.append((user_input, bot_response))
    return "", chat_history

with gr.Blocks() as demo:
    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="Type your message here...")
    clear = gr.ClearButton([msg, chatbot])

    msg.submit(chatbot_interface, [msg, chatbot], [msg, chatbot])

demo.launch()


Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://0d429beeca23d5f144.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
